# Drone Audio Classification â€” Kaggle Training

**Before running:**
- Settings > Accelerator > GPU T4 x2
- Settings > Internet > On
- Add dataset: `aranagnost/drone-audio-dataset`

## 1. Verify GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2. Clone the repo

In [ ]:
import os
import sys

REPO_DIR = '/kaggle/working/drone-audio-classification'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aranagnost/drone-audio-classification.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 3. Check dataset path

In [7]:
import os
print(os.listdir('/kaggle/input/datasets/aranagnost/drone-audio-dataset'))

['datasets']


In [9]:
DATASET_ROOT = '/kaggle/input/datasets/aranagnost/drone-audio-dataset/datasets/Drone_Audio_Dataset/audio'
print('Dataset root exists:', os.path.exists(DATASET_ROOT))
print('Subdirs:', os.listdir(DATASET_ROOT))

Dataset root exists: True
Subdirs: ['4_motors', '8_motors', '2_motors', 'not_a_drone', '6_motors']


## 4. Create artifacts directory

In [ ]:
os.makedirs('artifacts/checkpoints', exist_ok=True)
CACHE_DIR = '/kaggle/working/ast_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

## 5. Install transformers (for AST)

In [ ]:
!pip install -q transformers

## 6. Train AST v6 — 2 epochs (best was epoch 1 previously)

In [ ]:
os.environ['PYTHONPATH'] = REPO_DIR
!python training/train_ast.py --task stage2 --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --batch 16 --num_workers 4 --out artifacts/checkpoints/stage2_ast_v6.pt --freeze_encoder --unfreeze_top_n 3 --lr 5e-6 --dropout 0.3 --mlp_head --epochs 2 2>&1 | tee /tmp/ast_log.txt

## 7. Evaluate + Save + Send AST v6

In [ ]:
!python training/train_ast.py --task stage2 --eval --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --out artifacts/checkpoints/stage2_ast_v6.pt --mlp_head 2>&1 | tee /tmp/ast_eval_log.txt

## 8. Save checkpoint and send via Telegram

In [ ]:
import shutil, os, requests, subprocess
from kaggle_secrets import UserSecretsClient

CKPT_NAME = "stage2_ast_v6.pt"
src = f"artifacts/checkpoints/{CKPT_NAME}"
dst = f"/kaggle/working/{CKPT_NAME}"

def parse_best_info(log_path):
    try:
        with open(log_path) as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if '** Saved best' in line:
                epoch_line = lines[i-2].strip() if i >= 2 else ''
                cm_line    = lines[i-1].strip() if i >= 1 else ''
                saved_line = line.strip()
        return f"{epoch_line}
{cm_line}
{saved_line}"
    except Exception as e:
        return f"(could not parse log: {e})"

def parse_eval_info(log_path):
    try:
        with open(log_path) as f:
            lines = f.readlines()
        return ''.join(l for l in lines if 'macroF1' in l or 'F1(' in l or 'CM:' in l).strip()
    except Exception as e:
        return f"(could not parse eval log: {e})"

def upload_to_transfer_sh(filepath):
    try:
        result = subprocess.run(
            ["curl", "-s", "-F", f"file=@{filepath}", "https://transfer.sh/"],
            capture_output=True, text=True, timeout=1800
        )
        url = result.stdout.strip()
        return url if url.startswith("http") else f"(upload failed: {result.stdout} {result.stderr})"
    except Exception as e:
        return f"(upload failed: {e})"

best_info = parse_best_info('/tmp/ast_log.txt')
eval_info = parse_eval_info('/tmp/ast_eval_log.txt')

if not os.path.exists(src):
    print(f"[WARN] Checkpoint not found at {src}")
else:
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f"Saved {CKPT_NAME} ({size_mb:.1f} MB)")

    print("Uploading to transfer.sh...")
    download_url = upload_to_transfer_sh(dst)
    print(f"Download URL: {download_url}")

    secrets    = UserSecretsClient()
    TG_TOKEN   = secrets.get_secret("TG_TOKEN")
    TG_CHAT_ID = secrets.get_secret("TG_CHAT_ID")

    msg = (
        f"Checkpoint: {CKPT_NAME} ({size_mb:.1f} MB)
"
        f"Download: {download_url}

"
        f"Best epoch:
{best_info}

"
        f"Test eval:
{eval_info}"
    )
    r = requests.post(
        f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
        data={"chat_id": TG_CHAT_ID, "text": msg},
    )
    print("Sent to Telegram!" if r.ok else f"Failed: {r.text}")
    print("
Message sent:
" + msg)

## 9. Train BigCNN v1 — stage2 (up to 50 epochs)

In [ ]:
os.environ['PYTHONPATH'] = REPO_DIR
!python training/train_stage2.py --model big_cnn_v1 --dataset_root {DATASET_ROOT} --use_weighted_sampler --num_workers 4 --patience 8 --out artifacts/checkpoints/stage2_big_cnn_v1.pt 2>&1 | tee /tmp/bigcnn_log.txt

## 10. Evaluate BigCNN v1

In [ ]:
!python training/train_stage2.py --model big_cnn_v1 --eval --dataset_root {DATASET_ROOT} --out artifacts/checkpoints/stage2_big_cnn_v1.pt 2>&1 | tee /tmp/bigcnn_eval_log.txt

## 11. Save BigCNN checkpoint and send via Telegram

In [ ]:
import shutil, os, requests, subprocess
from kaggle_secrets import UserSecretsClient

CKPT_NAME = "stage2_big_cnn_v1.pt"
src = f"artifacts/checkpoints/{CKPT_NAME}"
dst = f"/kaggle/working/{CKPT_NAME}"

def parse_best_info(log_path):
    try:
        with open(log_path) as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if '** Saved best' in line:
                epoch_line = lines[i-2].strip() if i >= 2 else ''
                cm_line    = lines[i-1].strip() if i >= 1 else ''
                saved_line = line.strip()
        return f"{epoch_line}
{cm_line}
{saved_line}"
    except Exception as e:
        return f"(could not parse log: {e})"

def parse_eval_info(log_path):
    try:
        with open(log_path) as f:
            lines = f.readlines()
        return ''.join(l for l in lines if 'macroF1' in l or 'F1(' in l or 'CM [' in l).strip()
    except Exception as e:
        return f"(could not parse eval log: {e})"

def upload_to_transfer_sh(filepath):
    try:
        result = subprocess.run(
            ["curl", "-s", "-F", f"file=@{filepath}", "https://transfer.sh/"],
            capture_output=True, text=True, timeout=1800
        )
        url = result.stdout.strip()
        return url if url.startswith("http") else f"(upload failed: {result.stdout} {result.stderr})"
    except Exception as e:
        return f"(upload failed: {e})"

best_info = parse_best_info('/tmp/bigcnn_log.txt')
eval_info = parse_eval_info('/tmp/bigcnn_eval_log.txt')

if not os.path.exists(src):
    print(f"[WARN] Checkpoint not found at {src}")
else:
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f"Saved {CKPT_NAME} ({size_mb:.1f} MB)")

    print("Uploading to transfer.sh...")
    download_url = upload_to_transfer_sh(dst)
    print(f"Download URL: {download_url}")

    secrets    = UserSecretsClient()
    TG_TOKEN   = secrets.get_secret("TG_TOKEN")
    TG_CHAT_ID = secrets.get_secret("TG_CHAT_ID")

    msg = (
        f"Checkpoint: {CKPT_NAME} ({size_mb:.1f} MB)
"
        f"Download: {download_url}

"
        f"Best epoch:
{best_info}

"
        f"Test eval:
{eval_info}"
    )
    r = requests.post(
        f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
        data={"chat_id": TG_CHAT_ID, "text": msg},
    )
    print("Sent to Telegram!" if r.ok else f"Failed: {r.text}")
    print("
Message sent:
" + msg)